# Video-to-gaze pipeline

This notebook generates saliency latents for one input video, runs the bundled `final_model_90_45/checkpoint_70.pth`, and renders its predicted gaze trajectory over the source video. Run the cells from the repository root. Artifacts default to `artifacts/`; set `INFINITE_GAZE_ARTIFACT_ROOT` to place them elsewhere.

In [2]:
import os
from pathlib import Path
import sys

REPO = Path.cwd().resolve()
assert (REPO / 'generate_unisal_latents.py').exists(), 'Open this notebook from the repository root.'

# Change this to the video you want to process.
ARTIFACT_ROOT = Path(os.environ.get('INFINITE_GAZE_ARTIFACT_ROOT', REPO / 'artifacts')).expanduser().resolve()
INPUT_VIDEO = ARTIFACT_ROOT / 'datasets' / 'eval_videos' / '125.AVI'
assert INPUT_VIDEO.exists(), f'Video not found: {INPUT_VIDEO}'

PIPELINE_ROOT = ARTIFACT_ROOT / 'full_video_pipeline'
LATENT_ROOT = PIPELINE_ROOT / 'saliency_latents'
SAMPLE_ROOT = PIPELINE_ROOT / 'gaze_samples'
CHECKPOINT = REPO / 'final_model_90_45' / 'checkpoint_70.pth'
CONFIG = REPO / 'final_model_90_45' / 'inference_config.yaml'

assert CHECKPOINT.exists(), f'Checkpoint not found: {CHECKPOINT}'
assert CONFIG.exists(), f'Config not found: {CONFIG}'
print(f'Input: {INPUT_VIDEO}')
print(f'Outputs: {PIPELINE_ROOT}')

Input: C:\Users\jk8659\NYU\research\Scanpath\infinite-gaze-gen\artifacts\datasets\eval_videos\125.AVI
Outputs: C:\Users\jk8659\NYU\research\Scanpath\infinite-gaze-gen\artifacts\full_video_pipeline


In [3]:
# Generate saliency latents from the input video in this kernel.
from contextlib import contextmanager
import generate_unisal_latents

@contextmanager
def temporary_argv(args):
    original_argv = sys.argv
    sys.argv = [original_argv[0], *map(str, args)]
    try:
        yield
    finally:
        sys.argv = original_argv

CONDITIONING_DIR = LATENT_ROOT / INPUT_VIDEO.stem
with temporary_argv(['--video-path', INPUT_VIDEO, '--output-root', LATENT_ROOT, '--source', 'DHF1K']):
    generate_unisal_latents.main()

latent_files = sorted(CONDITIONING_DIR.glob('*.pt'))
assert latent_files, f'No latents were created in {CONDITIONING_DIR}'
print(f'Using {len(latent_files)} saliency latents from {CONDITIONING_DIR}')

torch device set to: cuda:0


Extracting 125: 100%|██████████| 847/847 [00:01<00:00, 426.69frame/s]


Extracted 847 frames to C:\Users\jk8659\NYU\research\Scanpath\infinite-gaze-gen\artifacts\full_video_pipeline\_saliency_unisal_workdir\125\images
Train ID: small_latent_unisal


c:\Users\jk8659\AppData\Local\anaconda3\envs\diffeye\lib\site-packages\torch\functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\TensorShape.cpp:4324.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Using device: cuda:0


Predicting 125:   0%|          | 0/847 [00:00<?, ?frame/s]

Best weights loaded

Warning! Evaluation bn source DIEM doesn't exist in model.
  Using DHF1K.


Predicting 125: 100%|██████████| 847/847 [00:17<00:00, 48.61frame/s]


Saved 847 pre-upsampling frame latents to C:\Users\jk8659\NYU\research\Scanpath\infinite-gaze-gen\artifacts\full_video_pipeline\saliency_latents\125
Done. Outputs saved under: C:\Users\jk8659\NYU\research\Scanpath\infinite-gaze-gen\artifacts\full_video_pipeline\_saliency_unisal_workdir\125
saved 847 latents to C:\Users\jk8659\NYU\research\Scanpath\infinite-gaze-gen\artifacts\full_video_pipeline\saliency_latents\125
latent shape: (64, 4, 6)
Using 847 saliency latents from C:\Users\jk8659\NYU\research\Scanpath\infinite-gaze-gen\artifacts\full_video_pipeline\saliency_latents\125


In [4]:
# Generate one deterministic gaze trajectory and render it as an overlay video.
import sample_video

sample_result = sample_video.sample_video(
    INPUT_VIDEO,
    conditioning_dir=CONDITIONING_DIR,
    config=CONFIG,
    checkpoint=CHECKPOINT,
    output_dir=SAMPLE_ROOT,
    num_samples=1,
    seed=12,
    overlay_trail=20,
)
print(f"Saved samples to {sample_result['output_dir']}")

c:\Users\jk8659\AppData\Local\anaconda3\envs\diffeye\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Saved samples to C:\Users\jk8659\NYU\research\Scanpath\infinite-gaze-gen\artifacts\full_video_pipeline\gaze_samples\125


In [5]:
from IPython.display import Video, display

OUTPUT_DIR = SAMPLE_ROOT / INPUT_VIDEO.stem / 'sample_000'
OVERLAY_VIDEO = OUTPUT_DIR / 'overlay.mp4'
SCANPATH_CSV = OUTPUT_DIR / 'scanpath.csv'

assert OVERLAY_VIDEO.exists(), f'Overlay was not created: {OVERLAY_VIDEO}'
print(f'Overlay video: {OVERLAY_VIDEO}')
print(f'Gaze coordinates: {SCANPATH_CSV}')


Overlay video: C:\Users\jk8659\NYU\research\Scanpath\infinite-gaze-gen\artifacts\full_video_pipeline\gaze_samples\125\sample_000\overlay.mp4
Gaze coordinates: C:\Users\jk8659\NYU\research\Scanpath\infinite-gaze-gen\artifacts\full_video_pipeline\gaze_samples\125\sample_000\scanpath.csv
